# Random Variables — Node 22

A **random variable** is a measurable function $X : \Omega \to \mathbb{R}$ from a probability space $(\Omega, \mathcal{F}, P)$ to the real line. The notation $\{X = 5\}$ is shorthand for the set $\{\omega \in \Omega : X(\omega) = 5\} \in \mathcal{F}$.

This notebook builds a die-roll RV, computes its empirical CDF, compares to the true CDF, and demonstrates that compositions $Y = g \circ X$ are again random variables. Standard library only (`random`, `collections`, `math`).

In [ ]:
import random
from collections import Counter
import math

random.seed(0)

# (Omega, F, P) — fair six-sided die
Omega = [1, 2, 3, 4, 5, 6]
P = {omega: 1.0/6.0 for omega in Omega}
print("Omega =", Omega)
print("P =", P)

## 1. Define the random variable $X : \Omega \to \mathbb{R}$

For a die-roll outcome $\omega$, we set $X(\omega) = \omega$ — the number shown. Since $\mathcal{F} = 2^\Omega$ is the full power set, *every* function $\Omega \to \mathbb{R}$ is measurable here, so $X$ is automatically a random variable.

In [ ]:
# The random variable X : Omega -> R
X = lambda omega: float(omega)

# Image (the range of X on Omega)
image_X = sorted({X(o) for o in Omega})
print("X(Omega) =", image_X)

# Verify measurability by checking a preimage explicitly
# {X <= 3} = {omega : X(omega) <= 3}
preimage = {omega for omega in Omega if X(omega) <= 3}
print("{X <= 3} =", preimage, "  P({X <= 3}) =", sum(P[o] for o in preimage))

## 2. True CDF $F_X(x) = P(X \le x)$

The CDF is the step function $F_X(x) = \lfloor x \rfloor / 6$ on $[1, 6]$, $0$ for $x < 1$, and $1$ for $x \ge 6$.

In [ ]:
def true_cdf(x):
    return sum(p for omega, p in P.items() if X(omega) <= x)

xs = [0.0, 0.5, 1.0, 2.5, 3.0, 4.5, 5.5, 6.0, 7.0]
print(f"{'x':>5} | {'F_X(x)':>8}")
print("-" * 18)
for x in xs:
    print(f"{x:>5.1f} | {true_cdf(x):>8.4f}")

## 3. Empirical CDF from $N = 20{,}000$ samples

Sample $\omega_1, \dots, \omega_N \sim P$, push through $X$, and form
$$\hat F_N(x) = \frac{1}{N} \sum_{i=1}^N \mathbf{1}\{X(\omega_i) \le x\}.$$
Glivenko–Cantelli says $\sup_x |\hat F_N(x) - F_X(x)| \to 0$ almost surely.

In [ ]:
N = 20_000
omegas = random.choices(Omega, weights=[P[o] for o in Omega], k=N)
X_samples = [X(o) for o in omegas]

def empirical_cdf(x, samples):
    return sum(1 for s in samples if s <= x) / len(samples)

print(f"{'x':>5} | {'true':>8} | {'emp.':>8} | {'|diff|':>8}")
print("-" * 40)
for x in xs:
    t = true_cdf(x)
    e = empirical_cdf(x, X_samples)
    print(f"{x:>5.1f} | {t:>8.4f} | {e:>8.4f} | {abs(t-e):>8.4f}")

## 4. Composition: $Y = X^2$ is also a random variable

If $X$ is a RV and $g : \mathbb{R} \to \mathbb{R}$ is Borel-measurable (continuous suffices), then $Y = g \circ X$ is a RV, because
$$(g \circ X)^{-1}(B) = X^{-1}(g^{-1}(B)) \in \mathcal{F}.$$
Here $g(x) = x^2$ is continuous, so $Y$ is a RV. Its image is $\{1, 4, 9, 16, 25, 36\}$, each occurring with probability $1/6$.

In [ ]:
g = lambda x: x * x
Y = lambda omega: g(X(omega))

print("Y(Omega) =", sorted({Y(o) for o in Omega}))

Y_samples = [Y(o) for o in omegas]
counts = Counter(Y_samples)
print(f"{'y':>4} | {'p_Y(y) true':>12} | {'p_Y(y) emp.':>12}")
print("-" * 38)
for y in sorted(counts):
    print(f"{y:>4.0f} | {1/6:>12.4f} | {counts[y]/N:>12.4f}")

## 5. Entropy of an RV — the ML connection

For a discrete RV $X$ with PMF $p$, the entropy
$$H(X) = -\sum_x p(x) \log p(x) = \mathbb{E}[-\log p(X)]$$
is itself the expectation of the random variable $-\log p(X)$. This is the same quantity that appears in Akgül et al. 2026, Eq. (1), for token-level entropy of an LLM's next-token distribution.

In [ ]:
def entropy(samples):
    counts = Counter(samples)
    n = len(samples)
    return -sum((c/n) * math.log(c/n) for c in counts.values())

H_X = entropy(X_samples)
H_Y = entropy(Y_samples)
H_true = math.log(6)   # uniform on 6 atoms
print(f"H(X) empirical = {H_X:.4f}")
print(f"H(Y) empirical = {H_Y:.4f}")
print(f"H(uniform-6)    = {H_true:.4f}   (true value for both)")

## 6. Takeaways

- A random variable is a **function** $\Omega \to \mathbb{R}$, not a number.
- $\{X \le x\}$ denotes a *set in* $\Omega$, and $F_X(x) = P(\{X \le x\})$.
- Borel-measurable compositions of RVs are RVs — sums, products, polynomials, $\log$, $\exp$ all stay inside the category.
- Every numerical quantity in ML (loss, gradient, reward, entropy) is implicitly an RV on some probability space.

**Next node:** [23-expectation](../23-expectation/) — integrating a random variable against $P$.